# Tutorial 05: Wikipedia API and data cleaning

Author: Maximilian Kreutner

In this notebook we will find all members of the current european parliament using wikidata.

Then, we will use the [python wrapper around the Wikipedia-API](https://github.com/martin-majlis/Wikipedia-API) to download their full biographies, clean the text, and extract their social media usernames.

In [ ]:
# Install dependencies
!pip install Wikipedia-API pandas requests tqdm beautifulsoup4

In [ ]:
# import dependencies
import time

import pandas as pd

import requests
import wikipediaapi

import re

from tqdm import tqdm

from urllib.parse import unquote, urljoin, urlparse

from bs4 import BeautifulSoup

## Using Wikidata to get a Clean List of Politicians

[Wikidata](https://www.wikidata.org/wiki/Wikidata:Main_Page) is the structured database behind Wikipedia. Instead of parsing a messy Wikipedia table, we can write a SPARQL query to ask Wikidata directly for: 

*"Give me all people whose position held is Member of the European Parliament for the 10th term, and give me their English Wikipedia article URL and optionally if they have their own website, their website url."*


If you want to try out more queries and examples, you can check the [official query page](https://query.wikidata.org/#).

In [22]:
# Define our User-Agent (Required by Wikimedia Foundation)
USER_AGENT = "DataCleaningClass (maximilian.kreutner@uni-mannheim.de)"

# SPARQL query to get current MEPs (10th term = Q114425478)
sparql_query = """
SELECT ?person ?personLabel ?article ?officialWebsite WHERE {
  ?person p:P39 ?statement .
  ?statement ps:P39 wd:Q27169 ;              # Member of the European Parliament
             pq:P2937 wd:Q114425478 .        # Tenth European Parliament

  OPTIONAL { ?person wdt:P856 ?officialWebsite . }   # official website
  ?article schema:about ?person ;
           schema:isPartOf <https://en.wikipedia.org/> .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". } # We only want the english articles
}
ORDER BY ?person
"""

title = "https://query.wikidata.org/sparql"
headers = {
    "User-Agent": USER_AGENT,
    "Accept": "application/sparql-results+json"
}

response = requests.get(
    title,
    headers=headers,
    params={"query": sparql_query, "format": "json"},
    timeout=60
)

response.raise_for_status()
data = response.json()
data

{'head': {'vars': ['person', 'personLabel', 'article', 'officialWebsite']},
 'results': {'bindings': [{'person': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q102395874'},
    'article': {'type': 'uri',
     'value': 'https://en.wikipedia.org/wiki/Zsuzsanna_Borvend%C3%A9g'},
    'personLabel': {'xml:lang': 'en',
     'type': 'literal',
     'value': 'Zsuzsanna Borvendég'}},
   {'person': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q10303913'},
    'article': {'type': 'uri',
     'value': 'https://en.wikipedia.org/wiki/Isilda_Gomes'},
    'personLabel': {'xml:lang': 'en',
     'type': 'literal',
     'value': 'Isilda Gomes'}},
   {'person': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q104070010'},
    'article': {'type': 'uri',
     'value': 'https://en.wikipedia.org/wiki/Claudiu_T%C3%A2rziu'},
    'personLabel': {'xml:lang': 'en',
     'type': 'literal',
     'value': 'Claudiu Târziu'}},
   {'person': {'type': 'uri',
     'value': 'h

We get back a json. Fill that data into a DataFrame which contains the columns: `name`, `official_website`, `wiki_title` and `URL`.
The wiki title is the last part of the URL after the last `/`.

In [23]:
mep_list = []
for item in data['results']['bindings']:
    name = item['personLabel']['value']
    # Extract the exact title from the URL and decode URL characters
    raw_url = item['article']['value']

    # Make sure to keep no official website in mind
    official_website = item.get("officialWebsite", {}).get("value")

    title = unquote(raw_url.split('/')[-1]) 
    mep_list.append({'name': name, "official_website":official_website , 'wiki_title': title, "URL": raw_url})

df = pd.DataFrame(mep_list)
display(df)

,name,official_website,wiki_title,URL
0,Zsuzsanna Borvendég,NaN,Zsuzsanna_Borvendég,https://en.wikipedia.org/wiki/Zsuzsanna_Borven...
1,Isilda Gomes,NaN,Isilda_Gomes,https://en.wikipedia.org/wiki/Isilda_Gomes
2,Claudiu Târziu,NaN,Claudiu_Târziu,https://en.wikipedia.org/wiki/Claudiu_T%C3%A2rziu
3,Emma Wiesner,NaN,Emma_Wiesner,https://en.wikipedia.org/wiki/Emma_Wiesner
4,Diana Șoșoacă,NaN,Diana_Șoșoacă,https://en.wikipedia.org/wiki/Diana_%C8%98o%C8...
...,...,...,...,...
743,Urmas Paet,http://urmaspaet.eu/,Urmas_Paet,https://en.wikipedia.org/wiki/Urmas_Paet
744,Villy Søvndal,NaN,Villy_Søvndal,https://en.wikipedia.org/wiki/Villy_S%C3%B8vndal
745,David McAllister,NaN,David_McAllister,https://en.wikipedia.org/wiki/David_McAllister
746,Thierry Mariani,https://www.mariani-2026.fr/,Thierry_Mariani,https://en.wikipedia.org/wiki/Thierry_Mariani


Let's do cleaning steps:

*Check for duplicates and remove them*

*Get insight into how many web adresses we have*

In [24]:
# Check for duplicates
print("Duplicate names:", df["name"].duplicated().sum())
print("Duplicate wiki titles:", df["wiki_title"].duplicated().sum())

Duplicate names: 15
Duplicate wiki titles: 15


In [25]:
df = df.drop_duplicates()

Check if we have an Wikipedia article and an official website for our politicians.

In [26]:
df.isna().sum()

name                  0
official_website    451
wiki_title            0
URL                   0
dtype: int64

## Fetch the actual full text articles

We can utilize `wikipediaapi` to get the full text of all the articles. For this we use the entries in `wiki_title` of our DataFrame.

To get the text of a wiki page we can use `Wikipedia.page(title)` and then `page.text`.

Save it in the DataFrame in the column `raw_text`.

We will limit the amount of politicians to 20, who have a website for this tutorial.

In [ ]:
df_with_website = df[df["official_website"].notna()].head(20).reset_index()

# Initialize the API
wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, language='en')

full_texts = []

for i, title in tqdm(enumerate(df_with_website["wiki_title"], start=1), total=len(df_with_website["wiki_title"])):
    try:
        page = wiki.page(title)

        if page.exists():
            full_texts.append(page.text)
        else:
            print("no page?")
            full_texts.append(None)

        # You should not do too many requests at once
        time.sleep(0.3)

    except requests.HTTPError as e:
        print(e)
        full_texts.append(None)
        time.sleep(2)

    except Exception:
        print("error")
        full_texts.append(None)
        time.sleep(1)

df_with_website["raw_text"] = full_texts
display(df_with_website)

100%|██████████| 20/20 [00:12<00:00,  1.58it/s]

Done!


,index,name,official_website,wiki_title,URL,raw_text
0,8,Mario Mantovani,http://www.mariomantovani.it,Mario_Mantovani,https://en.wikipedia.org/wiki/Mario_Mantovani,"Mario Mantovani (born 28 July 1950, in Arconat..."
1,15,Aldo Patriciello,http://www.patriciello.it,Aldo_Patriciello,https://en.wikipedia.org/wiki/Aldo_Patriciello,Aldo Patriciello (born 27 September 1957 in Ve...
2,17,Christine Schneider,https://www.christine-schneider.de/,Christine_Schneider,https://en.wikipedia.org/wiki/Christine_Schneider,Christine Schneider (born 5 June 1972) is a Ge...
3,19,Birgit Sippel,http://www.birgitsippel.de,Birgit_Sippel,https://en.wikipedia.org/wiki/Birgit_Sippel,Birgit Sippel (born 29 January 1960) is a Germ...
4,27,Biljana Borzan,http://www.biljanaborzan.eu/hr/,Biljana_Borzan,https://en.wikipedia.org/wiki/Biljana_Borzan,Biljana Borzan (née Čupurdija; born 29 Novembe...
5,28,René Repasi,https://repasi.eu/,René_Repasi,https://en.wikipedia.org/wiki/Ren%C3%A9_Repasi,René Repasi (born 8 November 1979) is a German...
6,30,Maria Noichl,http://maria-noichl.eu/,Maria_Noichl,https://en.wikipedia.org/wiki/Maria_Noichl,Maria Noichl (born 9 January 1967) is a German...
7,32,Jan-Christoph Oetjen,https://janchristoph-oetjen.europa.fdp.de/,Jan-Christoph_Oetjen,https://en.wikipedia.org/wiki/Jan-Christoph_Oe...,Jan-Christoph Oetjen (born 21 February 1978 in...
8,36,Marina Mesure,https://marinamesure.fr,Marina_Mesure,https://en.wikipedia.org/wiki/Marina_Mesure,Marina Mesure (born 12 July 1989) is a French ...
9,40,Matthias Ecke,https://matthias-ecke.eu/,Matthias_Ecke,https://en.wikipedia.org/wiki/Matthias_Ecke,Matthias Ecke (born 12 April 1983) is a German...


We still see some section headers that we would like to avoid, e.g. References.

In [32]:
for i in range(3):
    print("=" * 80)
    print(df_with_website.loc[i, "name"])
    print(df_with_website.loc[i, "raw_text"][:2000])
    print()

Mario Mantovani
Mario Mantovani (born 28 July 1950, in Arconate) is an Italian politician. He has been Member of the European Parliament, a mayor, a senator, the Undersecretary for Infrastructure and Transport under Berlusconi, and Vice-President of the Lombardy Region.  
In 2022, he was acquitted on appeal of corruption charges. In 2024, he was re-elected for a third mandate in the European Parliament, fifteen years after the end of his second mandate. 
His daughter is the deputy of the Brothers of Italy Lucrezia Mantovani.

Early life and career
Mantovani graduated in foreign languages and literature in 1975. From 1981 to 1986 he was director of the Padre Beccaro Institute in Milan.
In 1990 he founded the Sodalitas non-profit organization, which in Bellaria-Igea Marina has a series of residences and summer camps.
In 1996 he opened the Mantovani Foundation, specialized in the construction and management of nursing homes for the elderly. With his appointment as undersecretary in 2008 h

## Text Cleaning

The text usually contains artifacts that we don't want as a description of politicians, e.g. `References`, `External Links`, `See also`, `further reading` and `notes`.

We also should remove trailing whitespaces and normalize whitespaces that contain multiple characters.

Clean the text and safe it in the column `clean_text`.

In [39]:
import re
import pandas as pd

def clean_wiki_text(text):
    if pd.isna(text) or text is None:
        return None

    text = str(text)

    # normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # These are section we want to remove
    stop_sections = {
        "references",
        "external links",
        "see also",
        "further reading",
        "notes",
    }

    cleaned_lines = []
    for line in text.split("\n"):
        line_stripped = line.strip().lower()

        # remove surrounding "=" so "== References ==" becomes "references"
        heading = line_stripped.replace("=", "").strip()

        if heading in stop_sections:
            break

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    
    # collapse spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # remove spaces around newlines
    text = re.sub(r" *\n *", "\n", text)

    # collapse 3+ newlines to 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove trailing whitespaces
    text = text.strip()
    return text

df_with_website["clean_text"] = df_with_website["raw_text"].apply(clean_wiki_text)

In [41]:
for i in range(3):
    print("=" * 80)
    print(df_with_website.loc[i, "name"])
    print(df_with_website.loc[i, "clean_text"][:2000])
    print()

Mario Mantovani
Mario Mantovani (born 28 July 1950, in Arconate) is an Italian politician. He has been Member of the European Parliament, a mayor, a senator, the Undersecretary for Infrastructure and Transport under Berlusconi, and Vice-President of the Lombardy Region.
In 2022, he was acquitted on appeal of corruption charges. In 2024, he was re-elected for a third mandate in the European Parliament, fifteen years after the end of his second mandate.
His daughter is the deputy of the Brothers of Italy Lucrezia Mantovani.

Early life and career
Mantovani graduated in foreign languages and literature in 1975. From 1981 to 1986 he was director of the Padre Beccaro Institute in Milan.
In 1990 he founded the Sodalitas non-profit organization, which in Bellaria-Igea Marina has a series of residences and summer camps.
In 1996 he opened the Mantovani Foundation, specialized in the construction and management of nursing homes for the elderly. With his appointment as undersecretary in 2008 he l

## Find social media accounts on the official websites

We can use beautifulsoup and requests to find social media links on the official websites of the MEP.

Implement a method that takes the url and returns a dictionary with the `youtube`, `instagram`, `x_com` and `facebook` account of the politician.

In [42]:
def find_social_links(website_url):
    result = {
        "youtube": None,
        "instagram": None,
        "x_com": None,
        "facebook": None,
    }

    if website_url is None:
        return result

    website_url = str(website_url).strip()
    if website_url == "" or website_url.lower() == "nan":
        return result

    try:
        response = requests.get(website_url, headers=headers, timeout=15)
        response.raise_for_status()
    except Exception:
        return result

    soup = BeautifulSoup(response.text, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a.get("href", "").strip()
        if not href:
            continue

        href = urljoin(response.url, href)
        domain = urlparse(href).netloc.lower()

        if result["youtube"] is None and ("youtube.com" in domain or "youtu.be" in domain):
            result["youtube"] = href
        elif result["instagram"] is None and "instagram.com" in domain:
            result["instagram"] = href
        elif result["x_com"] is None and ("x.com" in domain or "twitter.com" in domain):
            result["x_com"] = href
        elif result["facebook"] is None and ("facebook.com" in domain or "fb.me" in domain):
            result["facebook"] = href

    return result

In [44]:
df_with_website[["youtube", "instagram", "x_com", "facebook"]] = (
    df_with_website["official_website"]
    .apply(find_social_links)
    .apply(pd.Series)
)

In [45]:
df_with_website

,index,name,official_website,wiki_title,URL,raw_text,clean_text,youtube,instagram,x_com,facebook
0,8,Mario Mantovani,http://www.mariomantovani.it,Mario_Mantovani,https://en.wikipedia.org/wiki/Mario_Mantovani,"Mario Mantovani (born 28 July 1950, in Arconat...","Mario Mantovani (born 28 July 1950, in Arconat...",NaN,NaN,NaN,NaN
1,15,Aldo Patriciello,http://www.patriciello.it,Aldo_Patriciello,https://en.wikipedia.org/wiki/Aldo_Patriciello,Aldo Patriciello (born 27 September 1957 in Ve...,Aldo Patriciello (born 27 September 1957 in Ve...,https://www.youtube.com/user/apatriciello,https://www.instagram.com/aldopatriciello_offi...,https://twitter.com/PatricielloAldo,https://www.facebook.com/AldoPatriciello
2,17,Christine Schneider,https://www.christine-schneider.de/,Christine_Schneider,https://en.wikipedia.org/wiki/Christine_Schneider,Christine Schneider (born 5 June 1972) is a Ge...,Christine Schneider (born 5 June 1972) is a Ge...,NaN,https://www.instagram.com/christineschneider_m...,NaN,https://www.facebook.com/christine.schneider.m...
3,19,Birgit Sippel,http://www.birgitsippel.de,Birgit_Sippel,https://en.wikipedia.org/wiki/Birgit_Sippel,Birgit Sippel (born 29 January 1960) is a Germ...,Birgit Sippel (born 29 January 1960) is a Germ...,https://www.youtube.com/user/birgitsippel,https://www.instagram.com/birgit_sippel/,https://twitter.com/BirgitSippelMEP,https://www.facebook.com/BirgitSippel
4,27,Biljana Borzan,http://www.biljanaborzan.eu/hr/,Biljana_Borzan,https://en.wikipedia.org/wiki/Biljana_Borzan,Biljana Borzan (née Čupurdija; born 29 Novembe...,Biljana Borzan (née Čupurdija; born 29 Novembe...,NaN,NaN,NaN,NaN
5,28,René Repasi,https://repasi.eu/,René_Repasi,https://en.wikipedia.org/wiki/Ren%C3%A9_Repasi,René Repasi (born 8 November 1979) is a German...,René Repasi (born 8 November 1979) is a German...,https://www.youtube.com/channel/UCYSfk8Sh3CCek...,https://www.instagram.com/rene.repasi/,https://twitter.com/repasi,https://www.facebook.com/repasi
6,30,Maria Noichl,http://maria-noichl.eu/,Maria_Noichl,https://en.wikipedia.org/wiki/Maria_Noichl,Maria Noichl (born 9 January 1967) is a German...,Maria Noichl (born 9 January 1967) is a German...,https://youtu.be/AgCIw7Oj5h0,NaN,https://twitter.com/marianoichl,https://www.facebook.com/noichl.eu
7,32,Jan-Christoph Oetjen,https://janchristoph-oetjen.europa.fdp.de/,Jan-Christoph_Oetjen,https://en.wikipedia.org/wiki/Jan-Christoph_Oe...,Jan-Christoph Oetjen (born 21 February 1978 in...,Jan-Christoph Oetjen (born 21 February 1978 in...,NaN,NaN,NaN,NaN
8,36,Marina Mesure,https://marinamesure.fr,Marina_Mesure,https://en.wikipedia.org/wiki/Marina_Mesure,Marina Mesure (born 12 July 1989) is a French ...,Marina Mesure (born 12 July 1989) is a French ...,https://www.youtube.com/@marina.mesure,https://www.instagram.com/marina.mesure/,https://twitter.com/MarinaMesure,https://www.facebook.com/MarinaMesureFi
9,40,Matthias Ecke,https://matthias-ecke.eu/,Matthias_Ecke,https://en.wikipedia.org/wiki/Matthias_Ecke,Matthias Ecke (born 12 April 1983) is a German...,Matthias Ecke (born 12 April 1983) is a German...,NaN,https://www.instagram.com/matthias.ecke/,https://twitter.com/MattEcke,https://www.facebook.com/matthiasecke


In [47]:
df_with_website.isna().sum()

index                0
name                 0
official_website     0
wiki_title           0
URL                  0
raw_text             0
clean_text           0
youtube             11
instagram            7
x_com                7
facebook             5
dtype: int64